In [0]:
# df = spark.table("silver.10000recipe.ingredient_master")
# df.repartition(10).write.mode("overwrite").option("header", "true").csv("/Volumes/silver/10000recipe/ingredient_master")

In [0]:
df_m = spark.table("silver.10000recipe.ingredient_master")
display(df_m)

In [0]:
df1 = spark.table("silver.ingredient.ingredient")
display(df1)

In [0]:
from pyspark.sql import functions as F

# 변수 재로드 (단독 실행 대비)
df = spark.table("silver.10000recipe.ingredient_master")
df1 = spark.table("silver.ingredient.ingredient")

# df에서 필요한 컨텍스트 컨럼만 추출
df_sel = df.select("ing_id", "std_name", "lv1", "lv2", "agro_category_ref").alias("a")

# df1에서 (카테고리, 재료명, 세부속성) 고유 조합
df1_keys = df1.select(
    F.col("카테고리"),
    F.col("재료명"),
    F.col("세부속성")
).distinct().alias("b")

# 조인 조건
join_cond = [
    F.col("a.agro_category_ref") == F.col("b.카테고리"),
    F.col("a.std_name") == F.col("b.재료명")
]

# 중복 항목 (df1에 존재) - inner join으로 세부속성 표시
dup_df = df_sel.join(df1_keys, join_cond, "inner").select(
    F.col("a.ing_id"), F.col("a.std_name"), F.col("a.lv1"), F.col("a.lv2"),
    F.col("a.agro_category_ref"),
    F.col("b.세부속성")
).distinct().orderBy("std_name")

# 비중복 항목 (df1에 없음) - left anti join
non_dup_df = df_sel.join(df1_keys, join_cond, "left_anti").select(
    F.col("a.ing_id"), F.col("a.std_name"), F.col("a.lv1"), F.col("a.lv2"),
    F.col("a.agro_category_ref")
).distinct().orderBy("std_name")

dup_cnt = dup_df.count()
non_dup_cnt = non_dup_df.count()

print(f"=== ✅ 중복 항목 (df1에 존재): {dup_cnt}건 ===")
display(dup_df)
print(f"\n=== ❌ 비중복 항목 (df1에 없음): {non_dup_cnt}건 ===")
display(non_dup_df)

In [0]:
df2 = spark.table("silver.10000recipe.crawl_target_ingredients")
display(df2)